In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Setting plot theme
sns.set_theme(style="whitegrid", palette="muted")

# Loading the SimScale DataSet
sim_df = pd.read_csv('simscale_combined_results.csv')

# Extracting velocity and CFD pressure drop (Delta P = P_face33 - P_face71)
v_sim = sim_df['Test_Velocity_V_test_ms'].values
dp_sim = sim_df['Pressure_Drop_Pa'].values

# Computing Analytical(MATLAB) Baseline
rho = 1.225  # Air density in kg/m^3
q_analytical = 0.5 * rho * (v_sim ** 2)

# Loading and Processing Experimental Data
exp1 = pd.read_csv('wind_tunnel_test_1.csv')
exp2 = pd.read_csv('wind_tunnel_test_2.csv')

# Concatenating both experimental test runs into a single dataframe
exp_combined = pd.concat([exp1, exp2], ignore_index=True)

# Grouping experimental data by FanPWM setting to compute steady-state means and standard deviations
exp_summary = exp_combined.groupby('FanPWM')['Pressure_Pa'].agg(['count', 'mean', 'std']).reset_index()
exp_filtered = exp_summary[exp_summary['count'] > 5].copy()  # Filter out low-sample transient noise

In [ ]:
# QUANTITATIVE ERROR ANALYSIS (CFD vs ANALYTICAL)
abs_error = np.abs(dp_sim - q_analytical)
pct_error = (abs_error / q_analytical) * 100

comparison_df = pd.DataFrame({
    'Velocity (m/s)': v_sim,
    'Analytical q (Pa)': q_analytical,
    'SimScale Delta P (Pa)': dp_sim,
    'Absolute Error (Pa)': abs_error,
    'Percentage Error (%)': pct_error
})

print("--- Comparative Summary Table (Head) ---")
print(comparison_df.head())

--- Comparative Summary Table (Head) ---
   Velocity (m/s)  Analytical q (Pa)  SimScale Delta P (Pa)  \
0             5.0            15.3125              40.541238   
1             6.0            22.0500              57.727745   
2             7.0            30.0125              77.781537   
3             8.0            39.2000             100.785252   
4             9.0            49.6125             126.754046   

   Absolute Error (Pa)  Percentage Error (%)  
0            25.228738            164.759107  
1            35.677745            161.803830  
2            47.769037            159.163804  
3            61.585252            157.105235  
4            77.141546            155.488125  


In [ ]:
# COMPARATIVE VISUALIZATIONS

# Plot 1: Analytical Baseline vs SimScale CFD Pressure Comparison
plt.figure(figsize=(10, 6), dpi=300)
plt.plot(v_sim, q_analytical, label='Analytical Baseline ($q = \\frac{1}{2}\\rho V^2$)', color='#1f77b4', linewidth=2.5, linestyle='--')
plt.plot(v_sim, dp_sim, label='SimScale CFD Parametric Sweep ($\\Delta P = P_{33} - P_{71}$)', color='#d62728', linewidth=2.5, marker='o', markersize=4)
plt.title('Low-Speed Wind Tunnel: Analytical vs. CFD Pressure Comparison', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Airflow Velocity ($m/s$)', fontsize=12, fontweight='semibold')
plt.ylabel('Pressure Drop / Dynamic Pressure ($Pa$)', fontsize=12, fontweight='semibold')
plt.legend(fontsize=11, frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.savefig('analytical_vs_cfd_pressure.png')
plt.close()

# Plot 2: Experimental Arduino Telemetry (Fan PWM vs Measured Pressure)
plt.figure(figsize=(10, 6), dpi=300)
plt.errorbar(exp_filtered['FanPWM'], exp_filtered['mean'], yerr=exp_filtered['std'],
             fmt='o-', color='#2ca02c', ecolor='#98df8a', elinewidth=1.5, capsize=3,
             label='Experimental Arduino DAQ (Pitot-Static System)')
plt.title('Experimental Wind Tunnel Telemetry: Fan PWM vs. Measured Differential Pressure', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Fan Speed Throttle (PWM: 0 - 255)', fontsize=12, fontweight='semibold')
plt.ylabel('Measured Differential Pressure ($\\Delta P_{exp}$ [Pa])', fontsize=12, fontweight='semibold')
plt.legend(fontsize=11, frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.savefig('experimental_telemetry_pressure.png')
plt.close()

print("\nPlots successfully generated and saved: 'analytical_vs_cfd_pressure.png' and 'experimental_telemetry_pressure.png'")


Plots successfully generated and saved: 'analytical_vs_cfd_pressure.png' and 'experimental_telemetry_pressure.png'


In [ ]:
# DERIVING EXPERIMENTAL VELOCITY FROM PITOT PRESSURE

# Converting measured experimental pressure to velocity using incompressible Bernoulli relation
valid_exp = exp_combined[exp_combined['Pressure_Pa'] > 0].copy()
valid_exp['Derived_Velocity_ms'] = np.sqrt((2 * valid_exp['Pressure_Pa']) / rho)

exp_vel_summary = valid_exp.groupby('FanPWM').agg({
    'Pressure_Pa': ['count', 'mean', 'std'],
    'Derived_Velocity_ms': ['mean', 'std']
}).reset_index()
exp_vel_summary.columns = ['FanPWM', 'Count', 'Mean_Pressure_Pa', 'Std_Pressure_Pa', 'Mean_Velocity_ms', 'Std_Velocity_ms']
exp_vel_filtered = exp_vel_summary[exp_vel_summary['Count'] > 5]

In [ ]:
# Plot 3: Analytical vs. Experimental Comparison
plt.figure(figsize=(10, 6), dpi=300)
plt.plot(v_sim, q_analytical, label='Analytical Baseline ($q = \\frac{1}{2}\\rho V^2$)', color='#1f77b4', linewidth=2.5, linestyle='--')
plt.errorbar(exp_vel_filtered['Mean_Velocity_ms'], exp_vel_filtered['Mean_Pressure_Pa'],
             xerr=exp_vel_filtered['Std_Velocity_ms'], yerr=exp_vel_filtered['Std_Pressure_Pa'],
             fmt='o', color='#2ca02c', ecolor='#98df8a', elinewidth=1.5, capsize=3,
             label='Experimental Arduino DAQ (Pitot-Static)')
plt.title('Analytical vs. Experimental Comparison: Velocity vs. Pressure', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Airflow Velocity ($m/s$)', fontsize=12, fontweight='semibold')
plt.ylabel('Pressure ($Pa$)', fontsize=12, fontweight='semibold')
plt.legend(fontsize=11, frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.savefig('analytical_vs_experimental.png')
plt.close()

In [ ]:
# Plot 4: CFD vs. Experimental Comparison
plt.figure(figsize=(10, 6), dpi=300)
plt.plot(v_sim, dp_sim, label='SimScale CFD Pressure Drop ($\\Delta P_{CFD}$)', color='#d62728', linewidth=2.5, marker='o', markersize=4)
plt.errorbar(exp_vel_filtered['Mean_Velocity_ms'], exp_vel_filtered['Mean_Pressure_Pa'],
             xerr=exp_vel_filtered['Std_Velocity_ms'], yerr=exp_vel_filtered['Std_Pressure_Pa'],
             fmt='o', color='#2ca02c', ecolor='#98df8a', elinewidth=1.5, capsize=3,
             label='Experimental Arduino DAQ ($\\Delta P_{exp}$)')
plt.title('CFD vs. Experimental Comparison: Pressure Drop vs. Velocity', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Airflow Velocity ($m/s$)', fontsize=12, fontweight='semibold')
plt.ylabel('Pressure Drop / Measured Pressure ($Pa$)', fontsize=12, fontweight='semibold')
plt.legend(fontsize=11, frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.savefig('cfd_vs_experimental.png')
plt.close()

print("\nSuccessfully generated and saved: 'analytical_vs_experimental.png' and 'cfd_vs_experimental.png'")


Successfully generated and saved: 'analytical_vs_experimental.png' and 'cfd_vs_experimental.png'
